# Gold-associated pixel spectral library: maximum-intensity wavelength extraction

**Purpose.** Extract the wavelength of maximum recorded intensity for each spectrum in a reference library derived from gold-associated pixels detected in hyperspectral cellular images. Each spectral column is treated as one spectrum.

**Input format.** A CSV or tab-delimited TXT file with wavelengths in the first column and spectral intensities in subsequent columns. **Output.** A CSV containing spectrum identifiers and peak wavelengths.

**Interpretation.** A spectral peak is not, by itself, a direct measurement of physical AuNP diameter.

## 1. Import libraries and configure file paths

Set the input file path and output directory. Paths are configurable and do not depend on the original researcher’s computer.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

INPUT_FILE = Path("data/reference_spectral_library.csv")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "reference_library_peak_wavelengths.csv"

## 2. Load and inspect the spectral library

Read CSV/TSV/TXT exports with delimiter and decimal-format checks. The first column must contain wavelength values (nm); all subsequent columns are independent spectra. Review the printed diagnostics before interpreting peak wavelengths.

In [ ]:
import csv

def load_spectral_library(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Spectral library not found: {path}")
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if path.suffix.lower() not in {".csv", ".tsv", ".txt"}:
        raise ValueError("Supported input formats: .csv, .tsv, .txt, .xlsx, .xls")

    # Compare candidate separators using the number of nonempty numeric wavelengths.
    # This avoids assuming that every TXT export is tab-delimited.
    candidates = []
    for sep in ("\t", ";", ","):
        for decimal in (".", ","):
            try:
                table = pd.read_csv(path, sep=sep, decimal=decimal, encoding="utf-8-sig", low_memory=False)
                if table.shape[1] < 2:
                    continue
                wave = pd.to_numeric(table.iloc[:, 0], errors="coerce")
                numeric_spectra = table.iloc[:, 1:].apply(pd.to_numeric, errors="coerce")
                score = (int(wave.notna().sum()), int(numeric_spectra.notna().sum().sum()))
                candidates.append((score, sep, decimal, table))
            except (UnicodeError, pd.errors.ParserError, ValueError):
                continue
    if not candidates:
        raise ValueError("Could not read wavelength and spectrum columns. Check delimiter and file encoding.")
    candidates.sort(key=lambda item: item[0], reverse=True)
    _, sep, decimal, table = candidates[0]
    print(f"Selected separator: {repr(sep)}; decimal symbol: {decimal}")
    return table

spectral_data = load_spectral_library(INPUT_FILE)
print("Raw dimensions:", spectral_data.shape)
print("First columns:", spectral_data.columns[:6].tolist())
print(spectral_data.iloc[:5, :min(5, spectral_data.shape[1])].to_string(index=False))


## 3. Identify the maximum-intensity wavelength for each spectrum

The maximum is calculated **within each original spectral column**, with no averaging, smoothing or peak fitting. Invalid spectra are reported rather than silently treated as zero. Peak intensity is exported alongside the peak wavelength for validation.

In [ ]:
def extract_maximum_intensity_wavelengths(data: pd.DataFrame) -> pd.DataFrame:
    wavelengths = pd.to_numeric(data.iloc[:, 0], errors="coerce")
    if wavelengths.notna().sum() < 3:
        raise ValueError("Wavelength axis is not numeric. Check the file separator, decimal symbol and header rows.")
    if not wavelengths.dropna().between(200, 2500).any():
        print("Warning: wavelength values are outside the usual optical range; check wavelength units.")
    records = []
    for j in range(1, data.shape[1]):
        spectrum_id = str(data.columns[j])
        intensity = pd.to_numeric(data.iloc[:, j], errors="coerce")
        valid = intensity.notna() & wavelengths.notna()
        if valid.any():
            peak_row = intensity.loc[valid].idxmax()
            peak_wavelength = float(wavelengths.loc[peak_row])
            peak_intensity = float(intensity.loc[peak_row])
        else:
            peak_wavelength = np.nan
            peak_intensity = np.nan
        records.append({"Spectrum_ID": spectrum_id, "Peak_Wavelength_nm": peak_wavelength,
                        "Peak_Intensity": peak_intensity, "Valid_Points": int(valid.sum())})
    return pd.DataFrame(records)

peak_results = extract_maximum_intensity_wavelengths(spectral_data)
print("Spectra:", len(peak_results))
print("Spectra with a detected maximum:", peak_results["Peak_Wavelength_nm"].notna().sum())
print("Wavelength range:", pd.to_numeric(spectral_data.iloc[:, 0], errors="coerce").agg(["min", "max"]).to_dict())
print("Detected peak range:", peak_results["Peak_Wavelength_nm"].agg(["min", "max"]).to_dict())
print(peak_results.head(15).to_string(index=False))
if peak_results["Peak_Wavelength_nm"].isna().all():
    raise ValueError("No peaks detected. Inspect the first rows and numeric formatting printed above.")


## 4. Visual validation of the detected peaks

Plot up to five raw spectra with their detected maxima. A peak marker must coincide with the highest recorded intensity for that spectrum. This diagnostic helps detect malformed input before exporting results.

In [ ]:
import matplotlib.pyplot as plt
wavelengths = pd.to_numeric(spectral_data.iloc[:, 0], errors="coerce")
fig, ax = plt.subplots(figsize=(10, 5))
for j in range(1, min(spectral_data.shape[1], 6)):
    intensities = pd.to_numeric(spectral_data.iloc[:, j], errors="coerce")
    valid = wavelengths.notna() & intensities.notna()
    if not valid.any():
        continue
    ax.plot(wavelengths[valid], intensities[valid], linewidth=1, label=str(spectral_data.columns[j]))
    peak_row = intensities.loc[valid].idxmax()
    ax.scatter([wavelengths.loc[peak_row]], [intensities.loc[peak_row]], s=45)
ax.set(xlabel="Wavelength (nm)", ylabel="Recorded intensity", title="Raw spectra and maximum-intensity wavelengths")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 5. Export the peak table

Export one result per original spectral column. No spectrum-level averaging is performed.

In [ ]:
peak_results.to_csv(OUTPUT_FILE, index=False)
print(f"Exported {len(peak_results)} spectra to: {OUTPUT_FILE}")

## Methodological notes

- The reference spectra originate from gold-associated pixels detected in cellular hyperspectral images, as described by the researcher.
- This notebook extracts maximum-intensity wavelengths but does **not** independently perform gold-pixel detection, spectral matching, or particle-size calibration.
- Wavelength shifts may also reflect aggregation or local optical environment; do not label the output as a measured particle-size distribution without separate calibration.